In [ ]:
!pip install -q beautifulsoup4 lxml requests langchain-text-splitters tiktoken

In [ ]:
import re
import json
import requests
import unicodedata
import hashlib
import tiktoken
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

URL_MAP = {

"https://www.iit.edu/registrar/registration":"registration",
"https://www.iit.edu/registrar/important-information":"registrar",
"https://www.iit.edu/registrar/policies-and-procedures":"policies",
"https://www.iit.edu/registrar/people":"registrar_people",
"https://www.iit.edu/registrar/registration/hold-information":"hold",
"https://www.iit.edu/registrar/students-and-alumni/transcripts":"transcripts",
"https://www.iit.edu/registrar/students-and-alumni":"alumni",
"https://www.iit.edu/commencement":"commencement",
"https://www.iit.edu/commencement/event-details-and-schedules":"events_and_schedules",
"https://www.iit.edu/financial-aid/policies-and-procedures/accelerated-masters-program-student-policy":"financial_aid_policies",
"https://www.iit.edu/directory?organization_type=16":"directory",
"https://www.iit.edu/coursera":"coursera",
"https://catalog.iit.edu/graduate/academic-policies-procedures/grades-transcripts/":"graduate_academic_policies",
"https://www.iit.edu/registrar/registration/registration-changes-add-drop-w-grades":"registration_add_drop",
"https://www.iit.edu/registrar/registration/withdrawal-course":"registration_withdrawal",
"https://www.iit.edu/registrar/policies-and-procedures/physical-or-financial-hardship-withdrawal-policy":"policies_hardship",
"https://www.iit.edu/registrar/students-and-alumni/grade-legend":"registrar_grade_legend",
"https://www.iit.edu/student-accounting/tuition-and-fees/mandatory-and-other-fees":"tuition",
"https://www.iit.edu/registrar/registration/course-repeat-policy":"registration_repeat",
"https://www.iit.edu/registrar/policies-and-procedures/withdrawing-vs-dropping-course":"registration_drop_vs_withdraw"
}

MAX_TOKENS = 350
OVERLAP_RATIO = 0.20

tokenizer = tiktoken.get_encoding("cl100k_base")

# ============================================================
# NAVIGATION FILTER (FIX ISSUE 7)
# ============================================================

NAVIGATION_PATTERNS = [
"skip to content",
"catalog home",
"iit home",
"apply now",
"search catalog",
"faculty a-z",
"programs",
"courses a-z",
"sitemap",
"archives",
"about",
"academics",
"admission",
"research",
"resources"
]

def is_navigation_text(text):

    t = text.lower()

    for n in NAVIGATION_PATTERNS:
        if n in t:
            return True

    return False


# ============================================================
# TOKEN COUNT
# ============================================================

def token_count(text):
    return len(tokenizer.encode(text))


# ============================================================
# NORMALIZE TEXT
# ============================================================

def normalize(text):

    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# ============================================================
# REMOVE NAVIGATION LISTS
# ============================================================

def is_navigation_list(text):

    words = text.split()

    if len(words) < 5:
        return False

    short_words = sum(1 for w in words if len(w) <= 6)

    if short_words / len(words) > 0.7:
        return True

    return False


# ============================================================
# REMOVE DUPLICATES IN BUFFER (FIX ISSUE 6)
# ============================================================

def remove_local_duplicates(lines):

    seen=set()
    unique=[]

    for line in lines:

        key=line.lower()

        if key not in seen:
            seen.add(key)
            unique.append(line)

    return unique


# ============================================================
# TABLE PARSER
# ============================================================

def parse_table(table):

    rows=[]

    header_row = table.find("tr")

    headers=[]

    if header_row:
        headers=[normalize(th.get_text(" ")) for th in header_row.find_all(["th","td"])]

    for tr in table.find_all("tr")[1:]:

        cols=[normalize(td.get_text(" ")) for td in tr.find_all(["td","th"])]

        if headers and len(cols)==len(headers):

            row=[]

            for h,v in zip(headers,cols):
                if v:
                    row.append(f"{h}: {v}")

            rows.append(" ".join(row))

        else:
            rows.append(" ".join(cols))

    return rows


# ============================================================
# LIST PARSER
# ============================================================

def parse_list(li):

    text = normalize(li.get_text(" "))

    if text and not is_navigation_list(text) and not is_navigation_text(text):
        return text

    return None


# ============================================================
# SENTENCE SPLIT
# ============================================================

def split_sentences(text):

    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [s.strip() for s in sentences if s.strip()]


# ============================================================
# CHUNKING
# ============================================================

def chunk_text(text):

    sentences = split_sentences(text)

    chunks=[]
    current=[]
    tokens=0

    for s in sentences:

        t = token_count(s)

        if tokens + t > MAX_TOKENS:

            chunk=" ".join(current)

            chunks.append(chunk)

            overlap_count=int(len(current)*OVERLAP_RATIO)

            current=current[-overlap_count:]

            tokens=token_count(" ".join(current))

        current.append(s)

        tokens+=t

    if current:
        chunks.append(" ".join(current))

    return chunks


# ============================================================
# CHUNK ID
# ============================================================

def make_chunk_id(prefix,url,index):

    url_hash = hashlib.md5(url.encode()).hexdigest()[:6]

    return f"{prefix}_{url_hash}_{index}"


# ============================================================
# EXTRACT PAGE
# ============================================================

def extract_page(url):

    prefix = URL_MAP[url]

    print("Processing:", url)

    r=requests.get(url,timeout=30)

    soup=BeautifulSoup(r.text,"lxml")

    for tag in soup(["nav","header","footer","aside","script","style"]):
        tag.decompose()

    for tag in soup.find_all(class_=re.compile("breadcrumb",re.I)):
        tag.decompose()

    main = soup.find("main") or soup.find("article") or soup.body

    topic = prefix.replace("_"," ").title()

    buffer=[]
    chunks=[]
    chunk_index=1

    def flush():

        nonlocal buffer,chunk_index

        buffer = remove_local_duplicates(buffer)

        text=normalize(topic + ". " + " ".join(buffer))

        buffer=[]

        if not text:
            return

        pieces = chunk_text(text)

        for p in pieces:

            if token_count(p) < 25:
                continue

            chunks.append({

                "chunk_id":make_chunk_id(prefix,url,chunk_index),
                "doc_type":"html",
                "doc_name":prefix,
                "source_url":url,
                "Topic":topic,
                "num_tokens":token_count(p),
                "content":p
            })

            chunk_index+=1


    for el in main.find_all(["h1","h2","h3","p","li","table"]):

        if el.name in ["h1","h2","h3"]:

            flush()

            new_topic=normalize(el.get_text(" "))

            if new_topic and not is_navigation_text(new_topic):
                topic=new_topic


        elif el.name=="p":

            txt=normalize(el.get_text(" "))

            if txt and not is_navigation_text(txt):
                buffer.append(txt)


        elif el.name=="li":

            txt=parse_list(el)

            if txt:
                buffer.append(txt)


        elif el.name=="table":

            rows=parse_table(el)

            for r in rows:
                buffer.append(r)
                flush()

    flush()

    return chunks


# ============================================================
# REFERENCE TEXT EXTRACTION
# ============================================================

def extract_reference_text(url):

    r=requests.get(url,timeout=30)

    soup=BeautifulSoup(r.text,"lxml")

    for tag in soup(["nav","header","footer","aside","script","style"]):
        tag.decompose()

    main=soup.find("main") or soup.find("article") or soup.body

    texts=[]

    for el in main.find_all(["p","li","td","th"]):

        txt=normalize(el.get_text(" "))

        if txt:
            texts.append(txt)

    return texts


# ============================================================
# SEMANTIC COVERAGE CHECK
# ============================================================

def semantic_coverage_check(url,page_chunks):

    original = extract_reference_text(url)

    # break chunks into sentences
    chunk_sentences = []

    for c in page_chunks:
        chunk_sentences.extend(split_sentences(c["content"]))

    missing = []

    for line in original:

        found = False

        for sent in chunk_sentences:

            # direct containment check first (fast)
            if line.lower() in sent.lower():
                found = True
                break

            # token overlap fallback
            line_tokens=set(line.lower().split())
            sent_tokens=set(sent.lower().split())

            overlap=len(line_tokens & sent_tokens)/max(len(line_tokens),1)

            if overlap > 0.7:
                found=True
                break

        if not found:
            missing.append(line)

    if missing:

        print("\n⚠ Possible missing content:",url)

        for m in missing[:5]:
            print("Missing:",m)


# ============================================================
# RUN PIPELINE
# ============================================================

all_chunks=[]

for url in URL_MAP:

    try:

        page_chunks = extract_page(url)

        semantic_coverage_check(url,page_chunks)

        all_chunks.extend(page_chunks)

    except Exception as e:

        print("ERROR:",url,e)

print("Total chunks:",len(all_chunks))


# ============================================================
# SAVE OUTPUT
# ============================================================

with open("final_chunks.json","w",encoding="utf-8") as f:

    json.dump(all_chunks,f,indent=2,ensure_ascii=False)

print("Saved final_chunks.json")

Processing: https://www.iit.edu/registrar/registration

⚠ Possible missing content: https://www.iit.edu/registrar/registration
Missing: Home
Missing: Information regarding registration policies and procedures can be found at the links below and on our Frequently Asked Questions page. Registration resources, including Student Dashboard and Registration Dashboard guides, can be found on our Resources page.
Missing: Taking a Course for Pass/Fail
Missing: Resources
Missing: If you have any questions, feel free to contact the Office of the Registrar at registrar@illinoistech.edu . Please remember to use your Illinois Tech email address and include your student number and full name when contacting our office.
Processing: https://www.iit.edu/registrar/important-information

⚠ Possible missing content: https://www.iit.edu/registrar/important-information
Missing: Home
Missing: Enrollment verifications are self-service through our secure website and may only be accessed by currently enrolled stu

In [ ]:
import re
import json
import requests
import unicodedata
import tiktoken
from bs4 import BeautifulSoup

# ============================================================
# CONFIG
# ============================================================

CHUNK_PREFIX = {

"https://www.iit.edu/registrar/registration": "registration",

"https://www.iit.edu/registrar/important-information": "registrar",

"https://www.iit.edu/registrar/policies-and-procedures": "policies",

"https://www.iit.edu/registrar/people": "registrar_people",

"https://www.iit.edu/registrar/registration/hold-information": "hold",

"https://www.iit.edu/registrar/students-and-alumni/transcripts": "transcripts",

"https://www.iit.edu/registrar/students-and-alumni": "Alumni",

"https://www.iit.edu/commencement": "commencement",

"https://www.iit.edu/commencement/event-details-and-schedules": "events_and_schedules",

"https://www.iit.edu/financial-aid/policies-and-procedures/accelerated-masters-program-student-policy":
"policies_accelerated_master"
}

tokenizer = tiktoken.get_encoding("cl100k_base")

# ============================================================
# TOKEN COUNT
# ============================================================

def token_count(text):
    return len(tokenizer.encode(text))

# ============================================================
# TEXT NORMALIZATION (FULL FIX)
# ============================================================

def normalize_text(text):

    if not text:
        return ""

    # normalize unicode encoding
    text = unicodedata.normalize("NFKC", text)

    # remove bullets completely
    text = re.sub(r'[\u2022\u2023\u25E6\u2043]', '', text)

    # remove markdown links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # collapse whitespace early
    text = re.sub(r'\s+', ' ', text).strip()

    # ============================================================
    # REMOVE BREADCRUMBS (CRITICAL FIX)
    # ============================================================

    breadcrumb_patterns = [
        r'^home\s+office',
        r'^home\s+registrar',
        r'^home\s+.*information',
        r'^home\s+.*registration',
        r'^home\s+.*transcripts',
        r'^home\s+.*policies',
        r'^home\s+.*commencement',
    ]

    text_lower = text.lower()

    for pattern in breadcrumb_patterns:

        if re.match(pattern, text_lower):

            # remove leading breadcrumb part
            parts = text.split(" ", 3)

            if len(parts) >= 3:
                text = " ".join(parts[3:])

            break

    # ============================================================

    replacements = {

        "\u2019": "'",
        "\u2018": "'",
        "\u201c": '"',
        "\u201d": '"',

        "\u2013": "-",
        "\u2014": "-",

        "\xa0": " ",
        "\u00a0": " ",
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# ============================================================
# TABLE PARSER (NO DATA LOSS)
# ============================================================

def parse_table(table):

    rows = []

    headers = []
    header_row = table.find("tr")

    if header_row:
        headers = [normalize_text(th.get_text(" ")) for th in header_row.find_all(["th","td"])]

    for tr in table.find_all("tr")[1:]:

        cols = [normalize_text(td.get_text(" ")) for td in tr.find_all(["td","th"])]

        if headers and len(cols)==len(headers):

            for h,v in zip(headers,cols):
                if v:
                    rows.append(f"{h}: {v}")

        else:
            rows.append(" ".join(cols))

    return rows


# ============================================================
# MAIN EXTRACTOR
# ============================================================

def extract_chunks(url):

    print("Processing:", url)

    prefix = CHUNK_PREFIX[url]

    response = requests.get(url)

    soup = BeautifulSoup(response.text, "lxml")

    # remove navigation
    for tag in soup(["nav","header","footer","aside","script","style"]):
        tag.decompose()

    # remove breadcrumb containers structurally
    for tag in soup.find_all(class_=re.compile("breadcrumb", re.I)):
        tag.decompose()

    main = soup.find("main")

    if not main:
        main = soup.find("article")

    if not main:
        main = soup.body

    chunks = []

    chunk_index = 1

    current_topic = prefix.replace("_"," ").title()

    current_content = []

    def save_chunk():

        nonlocal chunk_index, current_content

        content = " ".join(current_content)

        content = normalize_text(content)

        if len(content) < 30:
            current_content = []
            return

        chunks.append({

            "chunk_id": f"{prefix}_{chunk_index}",
            "doc_type": "html",
            "doc_name": prefix,
            "source_url": url,
            "Topic": current_topic,
            "page_start": None,
            "page_end": None,
            "page_range": None,
            "num_tokens": token_count(content),
            "content": content
        })

        chunk_index += 1
        current_content = []


    for element in main.descendants:

        if element.name in ["h1","h2","h3"]:

            save_chunk()

            topic = normalize_text(element.get_text(" "))

            if topic:
                current_topic = topic

        elif element.name == "p":

            text = normalize_text(element.get_text(" "))

            if text:
                current_content.append(text)

        elif element.name == "li":

            text = normalize_text(element.get_text(" "))

            if text:
                current_content.append(text)

        elif element.name == "table":

            rows = parse_table(element)

            current_content.extend(rows)

    save_chunk()

    return chunks


# ============================================================
# RUN ALL URLS
# ============================================================

all_chunks = []

for url in CHUNK_PREFIX:

    chunks = extract_chunks(url)

    all_chunks.extend(chunks)


print("Total chunks:", len(all_chunks))


# ============================================================
# SAVE JSON (NO UNICODE ESCAPE)
# ============================================================

with open("final_chunks.json", "w", encoding="utf-8") as f:

    json.dump(all_chunks, f, indent=2, ensure_ascii=False)


print("Saved to final_chunks.json")

Processing: https://www.iit.edu/registrar/registration
Processing: https://www.iit.edu/registrar/important-information
Processing: https://www.iit.edu/registrar/policies-and-procedures
Processing: https://www.iit.edu/registrar/people
Processing: https://www.iit.edu/registrar/registration/hold-information
Processing: https://www.iit.edu/registrar/students-and-alumni/transcripts
Processing: https://www.iit.edu/registrar/students-and-alumni
Processing: https://www.iit.edu/commencement
Processing: https://www.iit.edu/commencement/event-details-and-schedules
Processing: https://www.iit.edu/financial-aid/policies-and-procedures/accelerated-masters-program-student-policy
Total chunks: 49
Saved to final_chunks.json


In [ ]:
response = requests.get(url)

soup = BeautifulSoup(response.text, "lxml")

len(soup.find_all(["h1","h2","h3"]))

20

In [ ]:
for c in chunks:
    print(c["chunk_id"], c["num_tokens"])

policies_accelerated_master_1 216
policies_accelerated_master_2 78
policies_accelerated_master_3 17
policies_accelerated_master_4 125
policies_accelerated_master_5 125
policies_accelerated_master_6 116
policies_accelerated_master_7 110
policies_accelerated_master_8 52
policies_accelerated_master_9 434
policies_accelerated_master_10 39
policies_accelerated_master_11 235
policies_accelerated_master_12 239
